# 021-005. Debugging

*Reproduce the problem, inspect the relevant state, correct one cause, and test normal, boundary, and invalid inputs.*

The conceptual foundation is developed in Debugging, Imports, and Runtime Tools.

## Method · Calculation Trace

Printing a changing value is a simple way to locate the step where a result becomes unexpected.

> **Pause and predict**
>
> What should the final total be? Follow `total` once by hand before running the cell.

In [1]:
numbers = [10, 20, 30]
total = 0

for number in numbers:
    print("before:", total)
    total = total + number
    print("add:", number)
    print("after:", total)

print("final:", total)

before: 0
add: 10
after: 10
before: 10
add: 20
after: 30
before: 30
add: 30
after: 60
final: 60


> **Read the trace**
>
> The running total changes from 0 to 10, 30, and 60. If the final value were wrong, these intermediate values would narrow the search to one iteration.

## Implementation · Explicit Input Validation

An average is undefined for an empty list. The function checks that case and reports a specific error instead of allowing an unclear division error later. `raise ValueError(...)` stops the function with a precise message. The caller places the risky call under `try` and handles only that expected `ValueError` under `except`.

In [2]:
def average(numbers):
    if len(numbers) == 0:
        raise ValueError("numbers must contain at least one value")
    return sum(numbers) / len(numbers)

In [3]:
print("Average:", average([10, 20, 30]))

try:
    average([])
except ValueError as error:
    print("Invalid input:", error)

Average: 20.0
Invalid input: numbers must contain at least one value


The output preserves the valid average and labels the empty-list failure with the chosen message. Catching only `ValueError` allows unrelated faults to propagate; explicit validation suits input data, while `assert` remains a developer check.

## Implementation · Multiple Test Cases

One successful example is weak evidence. A small check set should include a typical case, a boundary case, and values with a different sign pattern. `assert condition, label` stops only when the condition is false; here it compares the actual result with the expected result.

In [4]:
test_cases = [
    {"name": "typical", "values": [10, 20, 30], "expected": 20},
    {"name": "one value", "values": [5], "expected": 5},
    {"name": "mixed signs", "values": [-2, 2], "expected": 0},
]

for case in test_cases:
    actual = average(case["values"])
    assert actual == case["expected"], case["name"]
    print(case["name"], "passed:", actual)

typical passed: 20.0
one value passed: 5.0
mixed signs passed: 0.0


> **Read the checks**
>
> Each assertion compares an actual result with an expected result. If a later edit breaks the function, the case name identifies which behavior changed.

## Implementation · Expected Conversion Error Handling

Real text data may contain a value that cannot be converted. Inside the loop, `try` contains the conversion, `except ValueError` handles an invalid string, and `else` runs only when conversion succeeds.

In [5]:
values = ["10", "20", "hello", "30"]

for value in values:
    try:
        number = int(value)
    except ValueError:
        print(repr(value), "was skipped")
    else:
        print(repr(value), "became", number)

'10' became 10
'20' became 20
'hello' was skipped
'30' became 30


The output shows three successful conversions and one skipped invalid string. Because `else` follows only a successful integer conversion, the result distinguishes valid and invalid text without hiding unrelated error types.

## Implementation · Function-Call Tracing in a Debugger

A debugger shows the same changing state without adding permanent print statements.

In [6]:
def double(number):
    result = number * 2
    return result

def triple(number):
    result = number * 3
    return result

In [7]:
def make_total(a, b):
    left = double(a)
    right = triple(b)
    return left + right

In [8]:
answer = make_total(4, 5)
print(answer)

23


### Interpretation · Function-Call Tracing in a Debugger

The call returns 23 because `double(4)` produces 8 and `triple(5)` produces 15. Inspecting those intermediate values separates a correct call path from an incorrect final sum.

**Debugger practice**

Put a breakpoint on `answer = make_total(4, 5)`. Use **Step Into** to enter `make_total()`, **Step Over** to run a call without entering it, and **Step Out** to return to the caller. Confirm that `left`, `right`, and `answer` become 8, 15, and 23.